# Dinámica molecular

_Física Computacional 2026_

> Notebook generado a partir de la clase `05-1-dinamica-molecular.qmd`. Ejecuten las celdas en orden.

> **Pregunta de la clase.** ¿Cómo se obtienen temperatura, presión y estructura
> de un fluido a partir de trayectorias de Newton, y cuándo un promedio temporal
> representa el equilibrio?

En la clase anterior tres cuerpos alcanzaron para perder la trayectoria después
de un horizonte de predicción. Un fluido tiene del orden de $10^{23}$
partículas: no hay ninguna esperanza de predecir trayectorias. Y sin embargo la
termodinámica del argón líquido es perfectamente predecible. Al terminar
podremos simular unos cientos de átomos de argón, medir su temperatura, su
presión, su estructura y su coeficiente de difusión, y decidir cuándo esos
promedios valen.

In [ ]:

import time

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib import animation


def animar(figura, actualizar, cuadros, intervalo=80):
    """Animación de matplotlib embebida en la página, sin depender de ffmpeg."""
    pelicula = animation.FuncAnimation(figura, actualizar, frames=cuadros, interval=intervalo)
    html = pelicula.to_jshtml(default_mode="loop")
    plt.close(figura)
    return HTML(html)

## 1. De tres cuerpos a un fluido

Las ecuaciones son las mismas de la clase 04: cada partícula obedece a Newton
con la suma de las fuerzas de todas las demás. La suma vectorizada con
`(N, N, dimensión)` también se reutiliza. Casi todo lo demás cambia.

| | Tres cuerpos (clase 04) | Dinámica molecular (hoy) |
|---|---|---|
| Interacción | gravedad, $1/r$: de largo alcance, siempre atractiva | Lennard-Jones: de corto alcance, con un carozo repulsivo |
| Encuentros cercanos | singulares, exigen paso adaptativo | imposibles: el carozo los impide |
| Integrador | DOP853, paso adaptativo | Verlet de la clase 02, paso fijo |
| Número de partículas | 3 | cientos o miles, representando $10^{23}$ |
| Borde | espacio abierto | caja periódica |
| Costo de la fuerza | irrelevante | lo que limita la simulación |
| Pregunta | una trayectoria | promedios: temperatura, presión, estructura |
| Caos | limita lo que se puede predecir | permite que una sola trayectoria explore los estados de equilibrio |

La última fila es el cambio de mirada. Ninguna trayectoria individual de la
simulación es la del argón real, ni hace falta. Lo que necesitamos es que una
trayectoria larga **recorra** los estados compatibles con la energía, de modo
que el promedio temporal coincida con el promedio sobre el ensamble. Esa
**hipótesis ergódica** no se demuestra en general: se pone a prueba. La clase
06 la va a contrastar con un promedio de ensamble calculado de otra manera.

## 2. Lennard-Jones y unidades reducidas

Dos átomos de gas noble se repelen a distancias cortas, por el principio de
exclusión, y se atraen débilmente a distancias mayores, por la interacción
entre dipolos inducidos. Lennard-Jones lo resume con dos parámetros:

$$
V(r)=4\varepsilon\left[\left(\frac\sigma r\right)^{12}-\left(\frac\sigma r\right)^6\right].
$$

Medimos longitudes en $\sigma$, energías en $\varepsilon$, masas en $m$ y
temperaturas en $\varepsilon/k_B$. El tiempo queda en
$\tau=\sigma\sqrt{m/\varepsilon}$. Para el argón:

| Magnitud | Unidad reducida | Argón |
|---|---|---|
| Longitud | $\sigma$ | 3.405 Å |
| Energía | $\varepsilon$ | $k_B\times119.8$ K |
| Tiempo | $\tau$ | 2.16 ps |
| Temperatura $T^*=1$ | $\varepsilon/k_B$ | 119.8 K |
| Densidad $\rho^*=0.8442$ | $\sigma^{-3}$ | 1.42 g/cm³ |

Un fluido simulado en unidades reducidas vale para cualquier gas noble que
Lennard-Jones describa bien: sólo cambian $\sigma$ y $\varepsilon$.

### 2.1. Radio de corte

El potencial cae como $r^{-6}$: en $r=2.5\sigma$ ya vale $-0.016\varepsilon$.
Lo **cortamos** en $r_c=2.5\sigma$ y lo **desplazamos** para que sea continuo
en el corte,

$$
V_c(r)=\begin{cases}V(r)-V(r_c) & r<r_c,\\ 0 & r\ge r_c.\end{cases}
$$

Sin el desplazamiento, cada vez que una pareja cruza $r_c$ la energía salta y
deja de conservarse. La fuerza sigue siendo discontinua en $r_c$, pero el salto
es chico.

In [ ]:

r = np.linspace(0.92, 3.0, 600)
V = 4 * (r**-12 - r**-6)
V_cortado = np.where(r < 2.5, V - 4 * (2.5**-12 - 2.5**-6), 0.0)
F = np.where(r < 2.5, 24 * (2 * r**-12 - r**-6) / r, 0.0)

fig, (ax_V, ax_F) = plt.subplots(1, 2, figsize=(9, 3.4))
ax_V.plot(r, V, label="completo")
ax_V.plot(r, V_cortado, "--", label="cortado y desplazado")
ax_V.axvline(2.5, color="0.6", lw=0.8)
ax_V.set(xlabel="$r/\\sigma$", ylabel="$V/\\varepsilon$", ylim=(-1.2, 1.5))
ax_V.legend()
recuadro = ax_V.inset_axes([0.5, 0.45, 0.45, 0.3])
recuadro.plot(r, V)
recuadro.plot(r, V_cortado, "--")
recuadro.set(xlim=(2.2, 3.0), ylim=(-0.03, 0.005))
recuadro.tick_params(labelsize=7)
ax_F.plot(r, F, color="C2")
ax_F.axhline(0, color="black", lw=0.8)
ax_F.axvline(2.5, color="0.6", lw=0.8)
ax_F.set(xlabel="$r/\\sigma$", ylabel="$F\\,\\sigma/\\varepsilon$", ylim=(-3, 5))
for ax in (ax_V, ax_F):
    ax.grid(alpha=0.25)
plt.show()

## 3. Caja y condiciones periódicas

Unos cientos de partículas en una caja con paredes serían casi todas
superficie. Para simular el interior de un fluido, la caja de lado $L$ se
repite en las tres direcciones: una partícula que sale por una cara entra por la
opuesta.

In [ ]:

fig, ax = plt.subplots(figsize=(5, 5))
azul, naranja = np.array([0.15, 0.55]), np.array([0.88, 0.45])
for i in (-1, 0, 1):
    for j in (-1, 0, 1):
        central = i == 0 and j == 0
        ax.add_patch(plt.Rectangle((i, j), 1, 1, fill=False, lw=2 if central else 0.8,
                                   color="black" if central else "0.6"))
        ax.plot(*(azul + [i, j]), "o", color="C0", alpha=1 if central else 0.35)
        ax.plot(*(naranja + [i, j]), "o", color="C1", alpha=1 if central else 0.35)
ax.annotate("", naranja - [1, 0], azul, arrowprops=dict(arrowstyle="<->", color="C2", lw=1.5))
ax.annotate("", naranja, azul, arrowprops=dict(arrowstyle="<->", color="0.5", ls="--"))
ax.add_patch(plt.Circle(azul, 0.4, fill=False, color="C0", ls=":"))
ax.text(azul[0] + 0.3, azul[1] + 0.3, "$r_c$", color="C0")
ax.set(xlim=(-1, 2), ylim=(-1, 2), aspect="equal", xticks=[], yticks=[])
plt.show()

Cada partícula interactúa con la **imagen más cercana** de cada una de las
otras. Con la diferencia $\mathbf d=\mathbf r_j-\mathbf r_i$, eso es restar el
múltiplo de $L$ más cercano en cada componente: `d -= L * np.round(d / L)`.
Para que ninguna partícula vea a la misma vecina dos veces, el corte tiene que
cumplir $r_c<L/2$.

In [ ]:

RC = 2.5  # radio de corte, en σ
V_RC = 4 * (RC**-12 - RC**-6)  # desplazamiento del potencial


def fuerzas(r, L):
    """Fuerzas de Lennard-Jones cortado y desplazado, con imagen mínima.

    Devuelve las fuerzas (N, 3), la energía potencial y el virial
    W = Σ_{i<j} r_ij · F_ij, que usaremos para la presión.
    """
    d = r[None, :, :] - r[:, None, :]
    d -= L * np.round(d / L)  # imagen mínima
    r2 = np.sum(d**2, axis=-1)
    np.fill_diagonal(r2, np.inf)
    dentro = r2 < RC**2
    inv6 = np.where(dentro, 1 / r2**3, 0.0)
    potencial = 0.5 * np.sum(np.where(dentro, 4 * (inv6**2 - inv6) - V_RC, 0.0))
    f_sobre_r2 = np.where(dentro, 24 * (2 * inv6**2 - inv6) / r2, 0.0)
    virial = 0.5 * np.sum(f_sobre_r2 * np.where(dentro, r2, 0.0))
    return -np.einsum("ij,ijk->ik", f_sobre_r2, d), potencial, virial

## 4. Del costo $N^2$ al costo $N$: listas de celdas

En la clase 04 sumar todas las parejas costaba $N^2$. Con un radio de corte,
cada partícula sólo interactúa con las que tiene a menos de $r_c$: unas
$\frac43\pi r_c^3\rho\approx55$ en un líquido, cualquiera sea $N$. El problema
es **encontrarlas** sin revisar todas las parejas.

La idea de las **listas de celdas**:

1. dividir la caja en $M^3$ celdas de lado $L/M\ge r_c$;
2. anotar en qué celda está cada partícula, con un costo proporcional a $N$;
3. buscar las vecinas de una partícula sólo en su celda y en las 26 que la
   rodean.

Cualquier vecina a menos de $r_c$ está forzosamente en esas 27 celdas. El
número de parejas revisadas es proporcional a $N$.

In [ ]:

rng_esquema = np.random.default_rng(5)
M_esquema, lado = 6, 1.0
puntos = rng_esquema.uniform(0, M_esquema, size=(150, 2))
centro = np.array([2.5, 3.4])
fig, ax = plt.subplots(figsize=(5, 5))
for i in range(M_esquema + 1):
    ax.axhline(i, color="0.7", lw=0.8)
    ax.axvline(i, color="0.7", lw=0.8)
celda = np.floor(centro).astype(int)
ax.add_patch(plt.Rectangle(celda - 1, 3, 3, color="C0", alpha=0.12))
cerca = np.all(np.abs(np.floor(puntos) - celda) <= 1, axis=1)
ax.plot(*puntos[~cerca].T, ".", color="0.6")
ax.plot(*puntos[cerca].T, "o", ms=4, color="C0")
ax.plot(*centro, "o", ms=7, color="C3")
ax.add_patch(plt.Circle(centro, lado, fill=False, color="C3"))
ax.set(xlim=(0, M_esquema), ylim=(0, M_esquema), aspect="equal", xticks=[], yticks=[])
plt.show()

In [ ]:

def fuerzas_celdas(r, L):
    """Las mismas fuerzas que `fuerzas`, buscando vecinas con listas de celdas.

    Las partículas se ordenan en un arreglo (M, M, M, K, 3), con K el máximo de
    partículas por celda y lugares vacíos marcados. Cada una de las 27 celdas
    vecinas se obtiene desplazando ese arreglo con np.roll. Requiere M ≥ 3.
    """
    M = int(L // RC)
    indice = np.floor(r / (L / M)).astype(int) % M
    celda = (indice[:, 0] * M + indice[:, 1]) * M + indice[:, 2]
    orden = np.argsort(celda, kind="stable")
    cuenta = np.bincount(celda, minlength=M**3)
    inicio = np.concatenate([[0], np.cumsum(cuenta)[:-1]])
    K = cuenta.max()
    lugar = np.arange(len(r)) - inicio[celda[orden]]
    tabla = np.full((M**3, K), -1)
    tabla[celda[orden], lugar] = orden
    tabla = tabla.reshape(M, M, M, K)
    ocupado = tabla >= 0
    P = np.where(ocupado[..., None], r[np.maximum(tabla, 0)], 0.0)

    F = np.zeros_like(P)
    potencial = virial = 0.0
    for desplazamiento in np.ndindex(3, 3, 3):
        corrimiento = tuple(1 - np.array(desplazamiento))
        Q = np.roll(P, corrimiento, axis=(0, 1, 2))
        ocupado_q = np.roll(ocupado, corrimiento, axis=(0, 1, 2))
        d = Q[..., None, :, :] - P[..., :, None, :]
        d -= L * np.round(d / L)
        r2 = np.sum(d**2, axis=-1)
        pareja = ocupado[..., :, None] & ocupado_q[..., None, :] & (r2 < RC**2)
        if corrimiento == (0, 0, 0):
            pareja &= ~np.eye(K, dtype=bool)  # sin autointeracción
        r2 = np.where(pareja, r2, 1.0)
        inv6 = np.where(pareja, 1 / r2**3, 0.0)
        potencial += 0.5 * np.sum(np.where(pareja, 4 * (inv6**2 - inv6) - V_RC, 0.0))
        f_sobre_r2 = np.where(pareja, 24 * (2 * inv6**2 - inv6) / r2, 0.0)
        virial += 0.5 * np.sum(f_sobre_r2 * r2)
        F -= np.einsum("...ij,...ijk->...ik", f_sobre_r2, d)

    fuerza = np.zeros_like(r)
    fuerza[tabla[ocupado]] = F[ocupado]
    return fuerza, potencial, virial


def red_fcc(n, L):
    """4 n³ partículas en una red cúbica centrada en las caras de lado L."""
    base = np.array([[0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]])
    celdas = np.array(list(np.ndindex(n, n, n)))
    return (celdas[:, None, :] + base[None, :, :]).reshape(-1, 3) * (L / n)

El primer control no depende de la física: con las mismas posiciones, las dos
funciones tienen que dar las mismas fuerzas hasta el redondeo.

In [ ]:

rng = np.random.default_rng(2026)
densidad = 0.8442


def configuracion(n):
    N = 4 * n**3
    L = (N / densidad) ** (1 / 3)
    return (red_fcc(n, L) + 0.05 * rng.normal(size=(N, 3))) % L, L


def tiempo(funcion, *argumentos):
    funcion(*argumentos)
    inicio = time.perf_counter()
    for _ in range(2):
        funcion(*argumentos)
    return (time.perf_counter() - inicio) / 2


costos = []
print(f"{'N':>6} {'máx |ΔF|':>9} {'directa (ms)':>13} {'celdas (ms)':>12}")
for n in (5, 6, 8, 10, 13):
    r, L = configuracion(n)
    directa = 1e3 * tiempo(fuerzas, r, L) if len(r) <= 2048 else np.nan
    diferencia = np.abs(fuerzas(r, L)[0] - fuerzas_celdas(r, L)[0]).max() if len(r) <= 2048 else np.nan
    celdas = 1e3 * tiempo(fuerzas_celdas, r, L)
    costos.append((len(r), directa, celdas))
    print(f"{len(r):6d} {diferencia:9.0e} {directa:13.1f} {celdas:12.1f}")
costos = np.array(costos)

In [ ]:

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.loglog(costos[:, 0], costos[:, 1], "o-", label="suma directa")
ax.loglog(costos[:, 0], costos[:, 2], "s-", label="listas de celdas")
ax.loglog(costos[:, 0], costos[-1, 2] * costos[:, 0] / costos[-1, 0], "k:", lw=1, label="$\\propto N$")
ax.set(xlabel="$N$", ylabel="ms por evaluación")
ax.legend()
ax.grid(alpha=0.25, which="both")
plt.show()

Con pocas partículas la suma directa gana: las celdas pagan el costo de
ordenar y de recorrer celdas casi vacías. Por encima de un millar de partículas
la diferencia crece con $N$. Para $N=8788$ no medimos la suma directa: el
arreglo de diferencias ocuparía unos 2 GB.

Dos comentarios:

- La **lista de Verlet** mejora esto: cada partícula guarda sus vecinas a menos
  de $r_c+\delta$ y la lista se reconstruye sólo cuando alguna se movió más de
  $\delta/2$.
- **Para la gravedad no sirve.** El aporte de una cáscara de radio $r$ es el
  número de partículas, que crece como $r^2$, por la fuerza de cada una. Con
  Lennard-Jones cae como $r^2\cdot r^{-7}=r^{-5}$ y el corte es una buena
  aproximación. Con la gravedad es $r^2\cdot r^{-2}$: todas las cáscaras pesan
  lo mismo.

Para el resto de la clase usamos $N=256$. La caja tiene $L=6.7\sigma$ y sólo
caben dos celdas por lado, así que la suma directa es la opción razonable.

## 5. Condiciones iniciales y equilibración

Arrancamos con las partículas en una red cúbica centrada en las caras, que es
la estructura cristalina del argón sólido, y velocidades gaussianas al azar.
Antes de integrar fijamos dos cosas:

- **momento total nulo**: si no, toda la caja deriva;
- **temperatura inicial** $T_0$, reescalando las velocidades una sola vez.

Por equipartición, la temperatura instantánea es

$$
T=\frac{2K}{3(N-1)},
$$

con $3(N-1)$ grados de libertad porque fijamos los tres componentes del momento
total. Integramos con el Verlet de velocidades de la clase 02.

In [ ]:

def simular(n, densidad, T0, dt, pasos, cada=10, semilla=2026):
    """Dinámica molecular sin termostato. Registra cada `cada` pasos.

    Devuelve un diccionario con t, cinética K, potencial U, virial W y las
    posiciones sin plegar a la caja, que necesitamos para la difusión.
    """
    rng = np.random.default_rng(semilla)
    N = 4 * n**3
    L = (N / densidad) ** (1 / 3)
    r = red_fcc(n, L)
    v = rng.normal(size=(N, 3))
    v -= v.mean(axis=0)
    v *= np.sqrt(3 * (N - 1) * T0 / np.sum(v**2))
    F, U, W = fuerzas(r, L)
    sin_plegar = r.copy()
    registro = {"t": [], "K": [], "U": [], "W": [], "r": []}
    for paso in range(pasos + 1):
        if paso % cada == 0:
            for nombre, valor in zip(registro, (paso * dt, 0.5 * np.sum(v**2), U, W, sin_plegar.copy())):
                registro[nombre].append(valor)
        v += 0.5 * dt * F
        r = (r + dt * v) % L
        sin_plegar += dt * v
        F, U, W = fuerzas(r, L)
        v += 0.5 * dt * F
    return {nombre: np.array(valores) for nombre, valores in registro.items()}, N, L


T0 = 1.5
registro, N, L = simular(n=4, densidad=densidad, T0=T0, dt=0.005, pasos=6000)
t, K, U, W = (registro[nombre] for nombre in ("t", "K", "U", "W"))
T = 2 * K / (3 * (N - 1))

**Antes de ver la figura:** arrancamos con un cristal perfecto a $T_0=1.5$.
¿Qué temperatura esperan una vez que el sistema se equilibra? ¿Mayor, igual o
menor que $T_0$?

In [ ]:

primeros = t <= 2
fig, (ax_T, ax_U) = plt.subplots(1, 2, figsize=(8, 3.2))
ax_T.plot(t[primeros], T[primeros], ".-", ms=3)
ax_T.set(xlabel="$t/\\tau$", ylabel="$T$ ($\\varepsilon/k_B$)")
ax_U.plot(t[primeros], U[primeros] / N, ".-", ms=3, color="C1")
ax_U.set(xlabel="$t/\\tau$", ylabel="$U/N$ ($\\varepsilon$)")
for ax in (ax_T, ax_U):
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

E = K + U
print(f"variación relativa de la energía total: {(E.max() - E.min()) / abs(E.mean()):.1e}")

En una fracción de $\tau$ la energía cinética se reparte con la potencial: el
cristal se funde y la temperatura cae. La energía total no cambia.

La animación muestra las 256 partículas proyectadas sobre el plano $xy$
durante los primeros $3\tau$. Al principio se ven las columnas de la red; en
menos de un $\tau$ desaparecen.

In [ ]:

posiciones = registro["r"] % L
fig, ax = plt.subplots(figsize=(4.5, 4.5))
nube = ax.plot([], [], "o", ms=5, alpha=0.6)[0]
reloj = ax.text(0.02, 1.02, "", transform=ax.transAxes)
ax.set(xlim=(0, L), ylim=(0, L), aspect="equal", xticks=[], yticks=[])


def cuadro(k):
    nube.set_data(posiciones[k, :, 0], posiciones[k, :, 1])
    reloj.set_text(f"t = {t[k]:.2f} τ,  T = {T[k]:.2f}")
    return nube, reloj


animar(fig, cuadro, cuadros=61)

¿Cuándo terminó la equilibración? No hay un criterio único. Algunos que se
usan juntos:

- las energías cinética y potencial fluctúan alrededor de un valor estable;
- el resultado no cambia si se descarta el doble de tiempo;
- la estructura inicial se perdió: lo veremos en la función de distribución
  radial.

Desde acá descartamos los primeros $10\tau$.

## 6. Observables

### 6.1. Temperatura y presión

La **temperatura** es el promedio temporal de la instantánea. La **presión** sale
del teorema del virial: al término de gas ideal se suma la contribución de las
fuerzas entre parejas,

$$
P=\rho T+\frac{1}{3V}\left\langle\sum_{i<j}\mathbf r_{ij}\cdot\mathbf F_{ij}\right\rangle .
$$

In [ ]:

equilibrio = t >= 10
V = L**3
P = N / V * T + W / (3 * V)
print(f"T   = {T[equilibrio].mean():.3f}   (desviación estándar {T[equilibrio].std():.3f})")
print(f"P   = {P[equilibrio].mean():.3f}   (desviación estándar {P[equilibrio].std():.3f})")
print(f"U/N = {U[equilibrio].mean() / N:.3f}")

Es la presión del potencial **cortado**. Para comparar con un experimento o con
el potencial completo hay que sumar una corrección por las parejas más allá de
$r_c$, que se calcula suponiendo que ahí el fluido es uniforme.

No damos todavía una barra de error. Las desviaciones estándar miden cuánto
fluctúa cada magnitud, no cuán bien conocemos su promedio. Volvemos sobre esto
en la sección 7.

### 6.2. Estructura: la función de distribución radial

La **función de distribución radial** $g(r)$ compara la densidad de partículas a
distancia $r$ de una dada con la densidad media. Se obtiene con un histograma
de las distancias entre parejas, normalizado por el número que habría en un
gas ideal en la misma cáscara:

$$
g(r)=\frac{\langle n(r)\rangle}{\frac12N\rho\,\frac43\pi\left[(r+\Delta r)^3-r^3\right]} .
$$

In [ ]:

def distribucion_radial(configuraciones, L, intervalos=150):
    bordes = np.linspace(0, L / 2, intervalos + 1)
    cuentas = np.zeros(intervalos)
    i, j = np.triu_indices(configuraciones.shape[1], k=1)
    for r in configuraciones:
        d = r[j] - r[i]
        d -= L * np.round(d / L)
        cuentas += np.histogram(np.linalg.norm(d, axis=1), bins=bordes)[0]
    n_particulas = configuraciones.shape[1]
    ideal = 0.5 * n_particulas * (n_particulas / L**3) * 4 / 3 * np.pi * np.diff(bordes**3)
    return 0.5 * (bordes[1:] + bordes[:-1]), cuentas / (len(configuraciones) * ideal)


liquido = registro["r"][equilibrio][::5] % L
r_g, g = distribucion_radial(liquido, L)
_, g_red = distribucion_radial(registro["r"][:1] % L, L)

In [ ]:

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(r_g, g_red / g_red.max() * g.max(), color="0.7", lw=0.8, label="red inicial (escala arbitraria)")
ax.plot(r_g, g, label="líquido")
ax.axhline(1, color="black", lw=0.8)
ax.set(xlabel="$r/\\sigma$", ylabel="$g(r)$", xlim=(0, L / 2))
ax.legend()
ax.grid(alpha=0.25)
plt.show()

Tres controles se leen en la figura:

- $g(r)=0$ por debajo de $r\approx0.9\sigma$: el carozo;
- el primer pico, cerca del mínimo del potencial en $2^{1/6}\sigma$, son las
  primeras vecinas;
- $g(r)\to1$ a distancias grandes: lejos de una partícula, el fluido no la ve.

Las líneas finas de la red se convirtieron en picos anchos que se amortiguan:
orden de corto alcance, sin orden de largo alcance. Eso es un líquido.

### 6.3. Dinámica: el desplazamiento cuadrático medio

Para medir cuánto se mueve una partícula necesitamos las posiciones **sin
plegar** a la caja: si no, una partícula que cruza una cara parece saltar una
distancia $L$.

$$
\langle\Delta r^2(t)\rangle=\frac1N\sum_i|\mathbf r_i(t_0+t)-\mathbf r_i(t_0)|^2\ \xrightarrow{\,t\to\infty\,}\ 6Dt .
$$

In [ ]:

r_eq = registro["r"][equilibrio]
tau = t[equilibrio] - t[equilibrio][0]
msd = np.mean(np.sum((r_eq - r_eq[0]) ** 2, axis=-1), axis=1)
tardio = tau >= 5
D = np.polyfit(tau[tardio], msd[tardio], 1)[0] / 6
print(f"D ≈ {D:.3f} σ²/τ  (ajuste lineal desde t = 5 τ, un solo origen temporal)")

In [ ]:

fig, ax = plt.subplots(figsize=(5, 5))
ax.add_patch(plt.Rectangle((0, 0), L, L, fill=False, color="0.6"))
for i in (3, 70, 140, 211):
    camino = r_eq[:, i, :2]
    ax.plot(*camino.T, lw=0.8)
    ax.plot(*camino[0], "k.", ms=6)
ax.set(aspect="equal", xlabel="$x/\\sigma$", ylabel="$y/\\sigma$")
ax.grid(alpha=0.25)
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.loglog(tau[1:], msd[1:], label="simulación")
ax.loglog(tau[1:10], 3 * T[equilibrio].mean() * tau[1:10] ** 2, "k--", lw=1, label="balístico, $3Tt^2$")
ax.set(xlabel="$t/\\tau$", ylabel="$\\langle\\Delta r^2\\rangle$ ($\\sigma^2$)")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

A tiempos cortos cada partícula vuela libre, con $\langle\Delta r^2\rangle=3Tt^2$.
A tiempos largos los choques la convierten en un caminante aleatorio y el
desplazamiento cuadrático crece linealmente. El coeficiente de difusión vuelve
a aparecer en la clase 07, con caminantes al azar, y en la 08, con la ecuación
del calor.

## 7. Promedios correlacionados

Registramos una muestra cada 10 pasos. ¿Son independientes? Medimos la
**autocorrelación** de la energía potencial, que compara la fluctuación en un
instante con la fluctuación un tiempo $s$ después:

$$
C(s)=\frac{\langle\delta U(t)\,\delta U(t+s)\rangle}{\langle\delta U^2\rangle},
\qquad \delta U=U-\langle U\rangle .
$$

In [ ]:

u = U[equilibrio] - U[equilibrio].mean()
retrasos = np.arange(40)
C = np.array([np.mean(u[: len(u) - k] * u[k:]) for k in retrasos]) / np.mean(u**2)
intervalo = t[1] - t[0]

In [ ]:

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(retrasos * intervalo, C, ".-", ms=3)
ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="$s/\\tau$", ylabel="$C(s)$")
ax.grid(alpha=0.25)
plt.show()

La correlación tarda varias muestras sucesivas en perderse; después, la curva
fluctúa alrededor de cero por el ruido estadístico. Tener 400 muestras no es
tener 400 mediciones independientes. Una barra de error calculada como si lo fueran es
demasiado chica. Cómo estimarla bien es tema de la clase 06 y del TP 05.

### Discusión en grupos: un resultado plausible

Un código de dinámica molecular **reescala las velocidades en cada paso** para
mantener la temperatura exactamente en $T=0.73$, también durante la medición. El
informe muestra una temperatura sin fluctuaciones, una función de distribución
radial razonable y un coeficiente de difusión.

1. ¿Qué cantidad de la sección 5 dejaría de conservarse? ¿Lo mostraría el
   informe?
2. ¿Qué observable de esta clase permitiría detectarlo sin leer el código?
3. ¿Cuál de los resultados del informe creen que sigue siendo válido, y cuál
   no?

## 8. Qué podemos concluir y qué no

Podemos concluir que:

- la misma suma de fuerzas de la clase 04, con imagen mínima y corte, simula un
  fluido;
- con un corte, las listas de celdas reducen el costo a $N$, y sólo valen para
  interacciones de corto alcance;
- la temperatura, la presión, $g(r)$ y $D$ salen de promedios sobre una
  trayectoria, y cada uno tiene un control propio;
- el cristal se funde y la temperatura de equilibrio no es la inicial.

No podemos concluir que:

- un promedio temporal sea el de equilibrio sin descartar la equilibración ni
  ponerlo a prueba;
- la desviación estándar de una serie sea la incertidumbre de su promedio;
- la presión del potencial cortado sea la del argón;
- 256 partículas representen el límite termodinámico sin variar $N$.

### Puente con el TP 05

En la práctica de hoy siguen con el TP 04. El TP 05, que empieza en la
práctica de la clase 06, retoma esta dinámica molecular: van a validarla,
estimar barras de error que tengan en cuenta la correlación y compararla con
un promedio de ensamble calculado con Metropolis. La
pregunta de salida:

> **¿Cuántas mediciones independientes hay en una trayectoria, y cómo cambia
> esa respuesta lo que podemos afirmar sobre el fluido?**